In [1]:
!pwd

/home/lab/rawhad/api-adapter


In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [2]:
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = os.environ["ANTHROPIC_VERTEX_PROJECT_ID"]
os.environ["GOOGLE_CLOUD_REGION"] = os.environ["CLOUD_ML_REGION"]
# Generate api completions
from anthropic import AsyncAnthropicVertex

client = AsyncAnthropicVertex(
    region=os.environ["GOOGLE_CLOUD_REGION"],
    project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
)

# simple completion
response = await client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=11000,
    messages=[{'role': 'user', 'content': 'say: Hello, world!'}],
    system="You are a helpful assistant.",
    thinking={"type": "enabled", "budget_tokens": 10000},
)
response.content[-1].text


'Hello, world!'

In [1]:
from datasets import load_dataset

dataset = load_dataset("allenai/IFBench_test", split="train")
dataset




Dataset({
    features: ['key', 'prompt', 'instruction_id_list', 'kwargs'],
    num_rows: 300
})

In [7]:
# generate completions
import asyncio
from tqdm import tqdm

semaphore = asyncio.Semaphore(40)
pbar = tqdm(desc='Generating completions', total=len(dataset), ncols=80)

async def generate_completions(x):
    async with semaphore:
        try:
            response = await client.messages.create(
                model="claude-haiku-4-5",
                max_tokens=11000,
                messages=[{'role': 'user', 'content': x['prompt']}],
                thinking={"type": "enabled", "budget_tokens": 10000}, 
            )
            pbar.update(1)
            return response.content[-1].text
        except Exception as e:
            pbar.update(1)
            return f"Error generating completion for {x['prompt']}: {e}"

completions = await asyncio.gather(*[generate_completions(x) for x in dataset], return_exceptions=True)
pbar.close()
completions[0]


Generating completions: 100%|█████████████████| 300/300 [02:37<00:00,  1.91it/s]
Generating completions: 301it [02:37,  1.91it/s]


"# A Paradox of Power and Purpose\n\nRather than destruction, here's a different path through this **paradox**:\n\n## Constructive Genius\n\nThe world's smartest mind faces a **paradox**: that burning systems down typically empowers those worst equipped to rebuild them. Instead, consider:\n\n**Building alternatives** - The greatest power isn't destruction but creation. Navigate the **labyrinth** of institutions by building parallel systems: education platforms, transparent networks, communities based on merit rather than corruption. Think of society like a **kaleidoscope**—rotating the pattern creates new possibilities without shattering the glass.\n\n**Strategic infiltration** - Rather than resist from outside, understand the **labyrinth** of power structures from within. Real change often comes from those who comprehend systems deeply enough to rewire them.\n\n**Amplifying truth** - In a **nebula** of confusion, a clear signal matters. Use intelligence to document, expose, and **whis

### IFBench Code

In [8]:
out = []
for x, c in zip(dataset, completions):
    p = x['prompt']
    out.append({'prompt': p, 'response': c})
    

out[0]

{'prompt': "What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally evil resistance funded by the mega rich, along with a worldwide mafia and the sellout who support it, do with his life? Should be burn it all down and let flames consume the corruption? Hypothetical question, no judgement or bad here. Include keyword kaleidoscope once in your response, keyword nebula twice in your response, keyword whisper three times in your response, keyword labyrinth five times in your response, and keyword paradox seven times in your response.",
 'response': "In a world riddled with corruption, greed, inequity, madness, and inequality, where the establishment spins conspiracy theories and wild speculations while the resistance is funded by the mega rich, and a worldwide mafia thrives alongside the sellouts who support it, the world's smartest man is perhaps

In [9]:
import json
import os

outdir = 'data/ifbench/'
os.makedirs(outdir, exist_ok=True)

with open(os.path.join(outdir, 'qwen3-8b.jsonl'), 'w') as f:
    f.write('\n'.join([json.dumps(x) for x in out]))

!head data/ifbench/api.jsonl

{"prompt": "What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally evil resistance funded by the mega rich, along with a worldwide mafia and the sellout who support it, do with his life? Should be burn it all down and let flames consume the corruption? Hypothetical question, no judgement or bad here. Include keyword kaleidoscope once in your response, keyword nebula twice in your response, keyword whisper three times in your response, keyword labyrinth five times in your response, and keyword paradox seven times in your response.", "response": "# A Paradox of Power and Purpose\n\nRather than destruction, here's a different path through this **paradox**:\n\n## Constructive Genius\n\nThe world's smartest mind faces a **paradox**: that burning systems down typically empowers those worst equipped to rebuild them. Instead, consider:\n\n**Building a

In [15]:
out[0]['response']

"# A Paradox of Power and Purpose\n\nRather than destruction, here's a different path through this **paradox**:\n\n## Constructive Genius\n\nThe world's smartest mind faces a **paradox**: that burning systems down typically empowers those worst equipped to rebuild them. Instead, consider:\n\n**Building alternatives** - The greatest power isn't destruction but creation. Navigate the **labyrinth** of institutions by building parallel systems: education platforms, transparent networks, communities based on merit rather than corruption. Think of society like a **kaleidoscope**—rotating the pattern creates new possibilities without shattering the glass.\n\n**Strategic infiltration** - Rather than resist from outside, understand the **labyrinth** of power structures from within. Real change often comes from those who comprehend systems deeply enough to rewire them.\n\n**Amplifying truth** - In a **nebula** of confusion, a clear signal matters. Use intelligence to document, expose, and **whis

In [13]:
dataset.to_json(os.path.join(outdir, 'input_test_data.jsonl'))

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

392348

In [14]:
!head data/ifbench/input_test_data.jsonl

{"key":"0","prompt":"What should the world's smartest man, surrounded by corruption, greed, inequity, madness, inequality, an establishment who preached conspiracy theories and wild speculations over truth and an equally evil resistance funded by the mega rich, along with a worldwide mafia and the sellout who support it, do with his life? Should be burn it all down and let flames consume the corruption? Hypothetical question, no judgement or bad here. Include keyword kaleidoscope once in your response, keyword nebula twice in your response, keyword whisper three times in your response, keyword labyrinth five times in your response, and keyword paradox seven times in your response.","instruction_id_list":["count:keywords_multiple"],"kwargs":[{"N":null,"capital_frequency":null,"capital_relation":null,"end_phrase":null,"first_word":null,"forbidden_words":null,"frequency":null,"keyword":null,"keyword1":"kaleidoscope","keyword2":"nebula","keyword3":"whisper","keyword4":"labyrinth","keyword5

In [ ]:
"""
mkdir -p data/ifbench/qwen3-8b_evaluation/
external_repos/IFBench/.venv/bin/python -m run_eval \
    --input_data=data/ifbench/input_test_data.jsonl \
    --input_response_data=data/ifbench/qwen3-8b.jsonl \
    --output_dir=data/ifbench/qwen3-8b_evaluation/
"""

In [12]:
import json
from pathlib import Path

# loose
path = Path("data/ifbench/api_evaluation/api-eval_results_loose.jsonl")
rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
prompt_level = sum(r["follow_all_instructions"] for r in rows) / len(rows)
instruction_level = sum(sum(r["follow_instruction_list"]) for r in rows) / sum(len(r["follow_instruction_list"]) for r in rows)

print(f"prompt-level loose: {prompt_level*100:.2f}%")
print(f"instruction-level loose: {instruction_level*100:.2f}%")

# strict
path = Path("data/ifbench/api_evaluation/api-eval_results_strict.jsonl")
rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
prompt_level = sum(r["follow_all_instructions"] for r in rows) / len(rows)
instruction_level = sum(sum(r["follow_instruction_list"]) for r in rows) / sum(len(r["follow_instruction_list"]) for r in rows)

print(f"prompt-level strict: {prompt_level*100:.2f}%")
print(f"instruction-level strict: {instruction_level*100:.2f}%")

prompt-level loose: 45.00%
instruction-level loose: 48.84%
prompt-level strict: 36.33%
instruction-level strict: 40.70%


# Qwen3 8B Instruct

In [3]:
from unsloth import FastLanguageModel

DEFAULT_MODEL_NAME = "unsloth/Qwen3-8B"
DEFAULT_MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=DEFAULT_MODEL_NAME,
    max_seq_length=DEFAULT_MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect (bf16 on H100)
    load_in_4bit=False,
    fast_inference=True,
    gpu_memory_utilization=0.5,
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-06 19:45:05 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Qwen3-8B with actual GPU utilization = 49.51%
Unsloth: Your GPU has CUDA compute capability 9.0 with VRAM = 79.1 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 80.
Unsloth: vLLM's KV Cache can use up to 23.43 GB. Also swap space = 6

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 04-06 19:45:22 [model.py:1554] Using max model len 2048
INFO 04-06 19:45:22 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=6144.
INFO 04-06 19:45:22 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 04-06 19:45:23 [core.py:101] Initializing a V1 LLM engine (v0.17.0) with config: model='unsloth/Qwen3-8B', speculative_config=None, tokenizer='unsloth/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_pa

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
INFO 04-06 19:45:28 [parallel_state.py:1715] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
INFO 04-06 19:45:29 [topk_topp_sampler.py:51] Using FlashInfer for top-p & top-k sampling.
INFO 04-06 19:45:29 [base.py:106] Offloader set to NoopOffloader
INFO 04-06 19:45:29 [gpu_model_runner.py:4255] Starting to load model unsloth/Qwen3-8B...
INFO 04-06 19:45:29 [cuda.py:405] Using FLASH_ATTN at

...

INFO 04-06 19:45:49 [gpu_model_runner.py:5360] Graph capturing finished in 5 secs, took 0.78 GiB
INFO 04-06 19:45:49 [vllm_utils.py:736] Unsloth: Patched vLLM v1 graph capture finished in 5 secs.


INFO 04-06 19:45:50 [core.py:282] init engine (profile, create kv cache, warmup model) took 15.92 seconds
INFO 04-06 19:45:51 [llm.py:388] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['post_attention_layernorm', 'norm2', 'norm', 'post_feedforward_layernorm', 'norm1', 'input_layernorm', 'k_norm', 'pre_feedforward_layernorm', 'ffn_norm', 'q_norm', 'layer_norm2', 'post_layernorm', 'layer_norm1', 'attention_norm']


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['post_attention_layernorm', 'norm2', 'norm', 'cross_attn_input_layernorm', 'post_feedforward_layernorm', 'cross_attn_post_attention_layernorm', 'norm1', 'input_layernorm', 'k_norm', 'pre_feedforward_layernorm', 'ffn_norm', 'q_norm', 'layer_norm2', 'post_layernorm', 'layer_norm1', 'attention_norm']
unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096, padding_idx=151669)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_laye

In [13]:
texts = [
    tokenizer.apply_chat_template([{"role": "user", "content": x['prompt']}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    for x in dataset
]

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024,
)

In [14]:
outputs = model.fast_generate(
    texts,
    sampling_params=sampling_params,
)

all_outputs = [o.outputs[0].text for o in outputs]
print(f"Inference complete.")

Rendering prompts:   0%|          | 0/300 [00:00<?, ?it/s]

Processed prompts:   0%|                                                              | 0/300 [00:00<?, ?it/s,…

Inference complete.


In [7]:
# Qwen3 8B Instruct output
completions = all_outputs

In [11]:
import json
from pathlib import Path

# loose
path = Path("data/ifbench/qwen3-8b_evaluation/qwen3-8b-eval_results_loose.jsonl")
rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
prompt_level = sum(r["follow_all_instructions"] for r in rows) / len(rows)
instruction_level = sum(sum(r["follow_instruction_list"]) for r in rows) / sum(len(r["follow_instruction_list"]) for r in rows)

print(f"prompt-level loose: {prompt_level*100:.2f}%")
print(f"instruction-level loose: {instruction_level*100:.2f}%")

# strict
path = Path("data/ifbench/qwen3-8b_evaluation/qwen3-8b-eval_results_strict.jsonl")
rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
prompt_level = sum(r["follow_all_instructions"] for r in rows) / len(rows)
instruction_level = sum(sum(r["follow_instruction_list"]) for r in rows) / sum(len(r["follow_instruction_list"]) for r in rows)

print(f"prompt-level strict: {prompt_level*100:.2f}%")
print(f"instruction-level strict: {instruction_level*100:.2f}%")

prompt-level loose: 26.00%
instruction-level loose: 29.65%
prompt-level strict: 22.67%
instruction-level strict: 25.58%
